In [19]:
from src.graph.graph_pipeline import build_pipeline, run_pipeline

pipe = build_pipeline()
frage = "In welcher Epoche haben Sensor 34, Sensor 8 gemeinsam gemessen?"
antwort = run_pipeline(pipe, frage)

print("Antwort des Systems:")
print(antwort)

PromptBuilder has 1 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


[QUERY_PROCESSOR:]
{'replies': ['MATCH (s1:Sensor {mote_id: 34})-[:GENERATED]->(m)-[:EPOCH_IN]->(e),\n      (s2:Sensor {mote_id: 8})-[:GENERATED]->(m)\nWHERE s1 <> s2\nRETURN DISTINCT e'], 'meta': [{'model': 'llama3.1:8b', 'created_at': '2025-12-04T14:22:27.063100362Z', 'done': True, 'total_duration': 16792511674, 'load_duration': 196349467, 'prompt_eval_duration': 6386952811, 'eval_duration': 10099599398, 'thinking': None, 'context': [128006, 882, 128007, 1432, 667, 1472, 527, 459, 6335, 304, 41744, 19, 73, 18221, 29182, 13, 4718, 3465, 374, 311, 7068, 7041, 832, 2764, 18221, 29182, 3319, 3196, 389, 279, 1217, 596, 3488, 382, 667, 30110, 44897, 50, 512, 667, 220, 16, 13, 8442, 8356, 264, 3254, 18221, 29182, 3319, 627, 667, 220, 17, 13, 5560, 1193, 49271, 11, 77332, 49271, 11, 5401, 11, 3651, 11, 2794, 11, 31980, 11, 15888, 7866, 11, 10592, 627, 667, 220, 18, 13, 3234, 539, 1005, 31876, 11, 73473, 11010, 11, 17640, 11, 9196, 11, 26836, 11, 39440, 28545, 11, 362, 2089, 34, 11, 477, 904,

In [ ]:
import json
from haystack.components.evaluators import FaithfulnessEvaluator
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from src.graph.graph_pipeline import build_pipeline

def evaluate_dataset(file_path="../src/graph/eval_dataset.jsonl"):
    """
    Liest Fragen aus einer JSONL-Datei, führt die GraphRAG-Pipeline aus
    und bewertet die Antworten mittels Faithfulness (Halluzinations-Check).
    """
    
    # 1. Pipeline laden
    rag_pipeline = build_pipeline()
    
    # 2. Evaluator initialisieren (Judge)
    # Nutzt ein LLM, um zu prüfen, ob die Antwort durch den Kontext gedeckt ist.
    # Wir nutzen OllamaChatGenerator, da der Evaluator intern 'messages' verwendet.
    evaluator = FaithfulnessEvaluator(
        chat_generator=OllamaChatGenerator(model="llama3.1:8b", url="http://ollama:11434")
    )
    
    results = []
    
    # 3. Dataset laden
    try:
        with open(file_path, "r") as f:
            dataset = [json.loads(line) for line in f]
    except FileNotFoundError:
        print(f"Datei nicht gefunden: {file_path}")
        return

    print(f"Starte Evaluation für {len(dataset)} Fragen...")
    print("-" * 60)

    for item in dataset:
        question = item["question"]
        q_id = item["id"]
        
        try:
            # A. Pipeline ausführen (Wir brauchen Antwort UND Kontext)
            # Da run_pipeline nur die Antwort gibt, rufen wir pipe.run direkt auf.
            # WICHTIG: include_outputs_from={"organizer"} sorgt dafür, dass wir den Kontext erhalten,
            # auch wenn er im Pipeline-Fluss konsumiert wird.
            result = rag_pipeline.run(
                {
                    "query_processor": {"query": question},
                    "generator": {"query": question}
                },
                include_outputs_from={"organizer"}
            )
            
            generated_answer = result["generator"]["answer"]
            retrieved_context = result["organizer"]["context"]
            
            # B. Evaluieren (Faithfulness)
            eval_result = evaluator.run(
                questions=[question],
                contexts=[[retrieved_context]], # Muss Liste von Listen sein
                predicted_answers=[generated_answer]
            )
            
            score = eval_result["results"][0]["score"]
            reasoning = eval_result["results"][0]["reasoning"]
            
            print(f"ID {q_id} | Score: {score:.2f} | Frage: {question[:40]}...")
            
            results.append({
                "id": q_id,
                "question": question,
                "answer": generated_answer,
                "faithfulness_score": score,
                "reasoning": reasoning
            })
            
        except Exception as e:
            print(f"Fehler bei ID {q_id}: {e}")
            # Fehler als 0.0 Score werten, damit sie in der Statistik auftauchen
            results.append({
                "id": q_id,
                "question": question,
                "answer": f"ERROR: {str(e)}",
                "faithfulness_score": 0.0,
                "reasoning": "Pipeline execution failed (likely invalid Cypher or connection issue)."
            })
            continue

    # 4. Zusammenfassung
    if results:
        avg_score = sum(r["faithfulness_score"] for r in results) / len(results)
        print("-" * 60)
        print(f"Durchschnittliche Faithfulness: {avg_score:.2f}")
        
        # Optional: Ergebnisse speichern
        with open("evaluation_results.json", "w") as f:
            json.dump(results, f, indent=2)
        print("Detaillierte Ergebnisse in 'evaluation_results.json' gespeichert.")

# Um die Evaluation zu starten, führe die folgende Zeile aus (ggf. Pfad anpassen):
evaluate_dataset()

PromptBuilder has 1 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
PromptBuilder has 3 prompt variables, but `

Starte Evaluation für 30 Fragen...
------------------------------------------------------------
[QUERY_PROCESSOR:]
{'replies': ['MATCH (s:Sensor {mote_id: 1})-[:GENERATED]->(m:Measurement)-[:AT_DATE]->(d:Date {date: "2004-03-31"})-[:HAS_HOUR]->(h:Hour {hour: 3})-[:HAS_MINUTE]->(mi:Minute {minute: 38})\nRETURN m.temperature AS temperature'], 'meta': [{'model': 'llama3.1:8b', 'created_at': '2025-12-04T15:35:29.528324524Z', 'done': True, 'total_duration': 24414312551, 'load_duration': 4685104039, 'prompt_eval_duration': 7617593105, 'eval_duration': 11978546705, 'thinking': None, 'context': [128006, 882, 128007, 1432, 667, 1472, 527, 459, 6335, 304, 41744, 19, 73, 18221, 29182, 13, 4718, 3465, 374, 311, 7068, 7041, 832, 2764, 18221, 29182, 3319, 3196, 389, 279, 1217, 596, 3488, 382, 667, 30110, 44897, 50, 512, 667, 220, 16, 13, 8442, 8356, 264, 3254, 18221, 29182, 3319, 627, 667, 220, 17, 13, 5560, 1193, 49271, 11, 77332, 49271, 11, 5401, 11, 3651, 11, 2794, 11, 31980, 11, 15888, 7866, 11,

  0%|          | 0/1 [01:40<?, ?it/s]



Fehler bei ID 1: Response from LLM evaluator is not a valid JSON.
